<a href="https://colab.research.google.com/github/Gelato1337/LLM-Workshop/blob/main/llm_information_extraction_workshop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM Information Extraction Workshop
## 1st pipeline:
**Two-Pass Discovery Method**

This notebook helps you extract structured information from text data using Large Language Models.

**The approach:**
1. **Pass 1 (Exploration):** Let the LLM freely discover patterns and categories in your data
2. **Pass 2 (Structured):** Use the discovered categories to do precise, consistent extraction

---

# 2nd pipeline:

Theme / keyword extraction - Summarization

## Step 0: Setup

Run this cell once to install and import everything needed.

In [ ]:
!apt-get install zstd

# Install Ollama (run once)
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama server in background
import subprocess
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

import time
time.sleep(3)  # Wait for server to start

print("✓ Ollama installed and running!")

In [ ]:
# Install dependencies
!pip install dspy datasets pymupdf python-docx thefuzz pandas tqdm -q
print("✓ Dependencies installed!")

# Imports
import dspy
import pandas as pd
import json
from pathlib import Path
from tqdm import tqdm
from collections import Counter
from pydantic import BaseModel
from thefuzz import fuzz
from IPython.display import display, Markdown
import requests                     # For talking to Ollama
print("✓ Imports ready!")
print("✓ All imports successful!")

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="pydantic")

In [ ]:
# Check GPU
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU available")

---
## Step 1: Download a Model

Choose a model based on available hardware:

- **Small (4GB):** `gemma3:4b`
- **Medium (14GB):** `gpt-oss:20b`
- **Large (48GB+):** `llama3.1:70b`

Full list: https://ollama.com/library

In [ ]:
# ============================================================
# CHANGE THIS: Select your model
# ============================================================

MODEL_NAME = "gpt-oss:20b"  # <-- Change to your preferred model

# ============================================================

# Download the model (only needed once per model)
!ollama pull {MODEL_NAME}

print(f"\n✓ Model {MODEL_NAME} ready!")

In [ ]:
# Configure DSPy
lm = dspy.LM(
    f"ollama_chat/{MODEL_NAME}",
    api_base="http://localhost:11434",
    max_tokens=150  # Increase to avoid truncation
)
dspy.configure(lm=lm)
print("✓ DSPy configured!")

test = dspy.Predict("question -> answer")

In [ ]:
response = test(question="Which of the following winter jackets brands gives the user more status, Moncler or Canada goose? Justification for your reasoning!!")
print(response.answer)

In [ ]:
import subprocess
import time

# Kill any existing ollama process and restart
subprocess.run(["pkill", "-f", "ollama"], capture_output=True)
time.sleep(1)

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

print("✓ Ollama server restarted!")

---
## Step 2: Load Your Data

**Option A:** Load from text files in a folder  
**Option B:** Load from a ready-made dataset (CSV, HuggingFace, etc.)

In [ ]:
def load_huggingface_dataset(dataset_name, split, text_column, n_samples=100):
    """Load dataset from HuggingFace."""
    from datasets import load_dataset

    ds = load_dataset(dataset_name, split=f"{split}[:{n_samples}]")
    df = pd.DataFrame({
        "id": [f"{dataset_name}_{i}" for i in range(len(ds))],
        "text": ds[text_column]
    })
    print(f"✓ Loaded {len(df)} samples from {dataset_name}")
    return df

print("✓ HuggingFace loader ready!")

# Document chunking for PDFs/DOCX

DIVIDER = "\n\n--- CHUNK ---\n\n"


def extract_text_from_file(file_path):
    """Extract text from PDF, DOCX, or TXT."""
    path = Path(file_path)

    if path.suffix.lower() == ".pdf":
        import pymupdf
        doc = pymupdf.open(path)
        return "\n\n".join(page.get_text() for page in doc)
    elif path.suffix.lower() == ".docx":
        from docx import Document
        doc = Document(path)
        return "\n\n".join(p.text for p in doc.paragraphs if p.text.strip())
    else:
        return path.read_text(encoding="utf-8")


def auto_chunk_to_txt(input_file, output_file, target_words=3000):
    """Auto-chunk document to editable TXT file."""
    text = extract_text_from_file(input_file)
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]

    chunks, current_chunk, current_words = [], [], 0

    for para in paragraphs:
        para_words = len(para.split())
        if current_words + para_words > target_words and current_chunk:
            chunks.append("\n\n".join(current_chunk))
            current_chunk, current_words = [para], para_words
        else:
            current_chunk.append(para)
            current_words += para_words

    if current_chunk:
        chunks.append("\n\n".join(current_chunk))

    Path(output_file).write_text(DIVIDER.join(chunks), encoding="utf-8")

    print(f"✓ Created {len(chunks)} chunks → {output_file}")
    print(f"\n→ Edit the file, then run load_chunked_txt()")


def load_chunked_txt(file_path):
    """Load edited chunk file as DataFrame."""
    text = Path(file_path).read_text(encoding="utf-8")
    chunks = [c.strip() for c in text.split("--- CHUNK ---") if c.strip()]

    df = pd.DataFrame({
        "id": [f"chunk_{i:03d}" for i in range(len(chunks))],
        "text": chunks
    })
    print(f"✓ Loaded {len(df)} chunks")
    return df

print("✓ Chunking functions ready!")



In [ ]:
# Load IMDB dataset
df = load_huggingface_dataset("imdb", "train", "text", n_samples=25000).sample(n=1000)
# Or load your own:
# df = load_chunked_txt("my_chunks.txt")

In [ ]:
df.head()

In [ ]:
# Preview first text
print(f"ID: {df.iloc[2]['id']}")
print(f"Length: {len(df.iloc[2]['text'])} chars")
print("="*50)
print(df.iloc[2]['text'][:1000] + "...")

# Select your own data:

In [ ]:
# --- OPTION 2: Raw File -> Auto-Chunker ---
import os
from google.colab import files

INPUT_FILE = "/content/source_document.pdf" # Change extension as needed
CHUNKED_FILE = "chunks_review.txt"

if not os.path.exists(INPUT_FILE):
    print("Please upload your source file:")
    uploaded = files.upload()
    uploaded_filename = list(uploaded.keys())[0]
    Path(INPUT_FILE).write_bytes(uploaded[uploaded_filename])

# Process using your existing chunker
auto_chunk_to_txt(INPUT_FILE, CHUNKED_FILE, target_words=400)

# Load and take a random sample
df_themes = load_chunked_txt(CHUNKED_FILE)
df_themes = df_themes.sample(n=min(1000, len(df_themes))).reset_index(drop=True)

df_themes.head()

In [ ]:
# --- OPTION 3: Pre-Chunked File ---
CHUNKED_FILE = "chunks_review.txt"

if os.path.exists(CHUNKED_FILE):
    df_themes = load_chunked_txt(CHUNKED_FILE)
    df_themes = df_themes.sample(n=min(1000, len(df_themes))).reset_index(drop=True)
    print("Ready with existing chunks.")
else:
    print("No chunk file found. Run Option 2 first.")
df_themes.head()

In [ ]:
#Data in correct order
df_themes = load_chunked_txt(CHUNKED_FILE).sort_values('id').head(1000)
df_themes.head()

# 1st Pipeline
## First pass extraction

In [ ]:
class Aspect(BaseModel):
    aspect: str      # Acting, plot, etc..
    sentiment: str   # positive, negative, neutral
    quote: str       # supporting quote - Reasoning

class AspectList(BaseModel):
    aspects: list[Aspect]

class ExtractAspects(dspy.Signature):
    """Extract aspects discussed in a review with sentiment and quotes."""
    text: str = dspy.InputField()
    aspects: AspectList = dspy.OutputField()

aspect_extractor = dspy.Predict(ExtractAspects)
print("✓ Extractor ready!")

In [ ]:
result = aspect_extractor(text=df.iloc[2]['text'][:2500])

print("EXTRACTED ASPECTS:")
for asp in result.aspects.aspects:
    print(f"  [{asp.sentiment:8}] {asp.aspect}")
    print(f"            \"{asp.quote}...\"\n")

In [ ]:
# ============================================================
# CHANGE THIS: Your extraction instructions
# ============================================================

ASPECT_INSTRUCTIONS = """
Extract all aspects the reviewer discusses (acting, plot, music, etc.).
For each: name the aspect, sentiment (positive/negative/neutral), and a quote.
"""

# ============================================================


# Preview prompt + data
df["prompt_preview"] = df["text"].apply(
    lambda x: f"{ASPECT_INSTRUCTIONS}\n\nTEXT TO PROCESS: \n \n{x[:2600]}..."
)

display(Markdown(df.iloc[2]["prompt_preview"]))

In [ ]:
df[["id", "prompt_preview"]].head(5)

In [ ]:
# ============================================================
N_SAMPLES_PASS1 = 20  # Start small
# ============================================================

pass1_results = []
all_aspects = []

for idx, row in tqdm(df.head(N_SAMPLES_PASS1).iterrows(), total=N_SAMPLES_PASS1):
    try:
        result = aspect_extractor(text=row["text"][:2500])
        for asp in result.aspects.aspects:
            pass1_results.append({"id": row["id"], **asp.dict()})
            all_aspects.append(asp.aspect.lower())
    except Exception as e:
        print(f"Error {row['id']}: {e}")

print(f"\n✓ Found {len(pass1_results)} aspects")

In [ ]:
print("DISCOVERED ASPECTS:")
for aspect, count in Counter(all_aspects).most_common(100):
    print(f"  {count:3d}x  {aspect}")

pd.DataFrame(pass1_results).head(100)

# Second pass extraction with fixed categories

In [ ]:
# ============================================================
# CHANGE THIS: Consolidate similar aspects into categories
# ============================================================

FINAL_CATEGORIES = [
    "Acting",
    "Plot",
    "Directing",
    "Cinematography",
    "Music",
    "Dialogue",
    "Pacing",
    "Emotional Impact",
    "Comedy",
    "Recommendation",
    "Movie quality"
]

# ============================================================

print("Categories:", FINAL_CATEGORIES)



In [ ]:
# Build prompt with the categories list
categories_formatted = "\n".join(f"- {cat}" for cat in FINAL_CATEGORIES)


CUSTOM_PROMPT = f"""Extract aspects from a movie review.

ALLOWED CATEGORIES - Use ONLY these exact names:
{categories_formatted}

For each aspect found:
- category: MUST be one of the categories listed above (exactly as written)
- sentiment: positive, negative, or neutral (lowercase only)
- quote: short quote from the text supporting this aspect

If something doesn't fit the categories above, use "Other"."""

In [ ]:
class StructuredAspect(BaseModel):
    category: str
    sentiment: str
    quote: str


class ExtractStructured(dspy.Signature):
    __doc__ = CUSTOM_PROMPT

    text: str = dspy.InputField(desc="The review text to analyze")
    aspects: list[StructuredAspect] = dspy.OutputField(desc="List of extracted aspects")

structured_extractor = dspy.Predict(ExtractStructured)

# Verify prompt
print("PROMPT:")
print(CUSTOM_PROMPT)

In [ ]:
result = structured_extractor(text=df.iloc[0]['text'][:2500])

for asp in result.aspects:
    print(f"[{asp.sentiment:8}] {asp.category}: \"{asp.quote[:50]}...\"")

In [ ]:
# After a call, inspect history
print(lm.history[-1])

In [ ]:
# ============================================================
N_SAMPLES_PASS2 = 15
# ============================================================

pass2_results = []

for idx, row in tqdm(df.head(N_SAMPLES_PASS2).iterrows(), total=N_SAMPLES_PASS2):
    try:
        result = structured_extractor(text=row["text"][:2500])
        for asp in result.aspects:
            pass2_results.append({
                "doc_id": row["id"],
                "category": asp.category,
                "sentiment": asp.sentiment,
                "quote": asp.quote
            })
    except Exception as e:
        print(f"Error {row['id']}: {e}")

print(f"\n✓ Extracted {len(pass2_results)} aspects")

results_df = pd.DataFrame(pass2_results)
results_df.head(10)

In [ ]:
results_df = pd.DataFrame(pass2_results)
results_df.head(50)



In [ ]:
print("CATEGORY DISTRIBUTION:")
print(results_df["category"].value_counts())

In [ ]:
results_df.to_csv("aspect_results.csv", index=False)
print("✓ Saved: aspect_results.csv")

 # Pipeline 2 - Thematic Summarization

In [ ]:
class Theme(BaseModel):
    theme: str
    summary: str
    quote: str          # EXACT quote from source
    keywords: list[str]

THEME_PROMPT = """Extract themes from this text.

For each theme provide:
- theme: short name for the theme
- summary: 1-2 sentence summary
- quote: EXACT word-for-word quote from the text (copy directly)
- keywords: list of 3-5 relevant keywords

Return as JSON array. Quotes must be EXACT copies from the source text."""

class ExtractThemes(dspy.Signature):
    __doc__ = THEME_PROMPT

    text: str = dspy.InputField(desc="The text to analyze")
    themes: list[Theme] = dspy.OutputField(desc="List of themes")

theme_extractor = dspy.Predict(ExtractThemes)
print("✓ Theme extractor ready!")

In [ ]:
result = theme_extractor(text=df_themes.iloc[50]['text'])

for t in result.themes:
    print(f"\nTheme: {t.theme}")
    print(f"Summary: {t.summary}")
    print(f"Quote: \"{t.quote}\"")
    print(f"Keywords: {t.keywords}")

In [ ]:
THEME_INSTRUCTIONS = """
Extract main themes. For each:
- Theme name
- Brief summary
- EXACT quote (word-for-word from text)
- Keywords
"""

df_themes["prompt_preview"] = df_themes["text"].apply(
    lambda x: f"{THEME_INSTRUCTIONS}\n\nTEXT:\n{x[:400]}..."
)
df_themes[["id", "prompt_preview"]].head(5)

In [ ]:
theme_results = []

for idx, row in tqdm(df_themes.head().iterrows(), total=20):
    try:
        result = theme_extractor(text=row["text"])
        for t in result.themes:
            theme_results.append({
                "chunk_id": row["id"],
                "source_text": row["text"],
                "theme": t.theme,
                "summary": t.summary,
                "quote": t.quote,
                "keywords": ", ".join(t.keywords)
            })
    except Exception as e:
        print(f"Error {row['id']}: {e}")

print(f"\n✓ Found {len(theme_results)} themes")

In [ ]:
# After a call, inspect history
print(lm.history[-1])

In [ ]:
themes_df = pd.DataFrame(theme_results)
themes_df.head(50)

In [ ]:
def find_best_match(quote, source, threshold=90):
    """Find best matching substring using fuzzy matching."""
    q = quote.lower().strip()
    s = source.lower()

    # Exact match
    if q in s:
        return True, 100, quote

    # Sliding window fuzzy match
    best_ratio, best_match = 0, ""
    q_len = len(q)

    for win in [q_len, int(q_len*0.8), int(q_len*1.2)]:
        if win <= 0: continue
        for i in range(0, max(1, len(s)-win+1), 15):
            window = s[i:i+win]
            ratio = fuzz.ratio(q, window)
            if ratio > best_ratio:
                best_ratio = ratio
                best_match = source[i:i+win]

    return best_ratio >= threshold, best_ratio, best_match


def validate_quotes(df, threshold=90):
    """Validate all quotes against source text."""
    results = []

    for _, row in tqdm(df.iterrows(), total=len(df)):
        valid, ratio, matched = find_best_match(row["quote"], row["source_text"], threshold)
        results.append({"quote_valid": valid, "match_ratio": ratio, "matched_text": matched})

    for col in ["quote_valid", "match_ratio", "matched_text"]:
        df[col] = [r[col] for r in results]
    return df

print("✓ Validation ready!")

In [ ]:
# ============================================================
MATCH_THRESHOLD = 90  # Minimum similarity %
# ============================================================

themes_df = validate_quotes(themes_df, threshold=MATCH_THRESHOLD)

# Show invalid quotes
invalid = themes_df[~themes_df["quote_valid"]]

if len(invalid) > 0:
    print(f"INVALID QUOTES ({len(invalid)}):")
    for _, row in invalid.head(5).iterrows():
        print(f"\n[{row['chunk_id']}] {row['match_ratio']}%")
        print(f"LLM:   \"{row['quote'][:80]}...\"")
        print(f"Found: \"{row['matched_text'][:80]}...\"")
else:
    print("✓ All quotes valid!")

In [ ]:
# All results
themes_df[["chunk_id", "theme", "summary", "quote", "keywords",
           "quote_valid", "match_ratio"]].to_csv("theme_results.csv", index=False)
print("✓ Saved: theme_results.csv")